# **Predictions (visual transformers)**

## Navigation

- [**Basic imports and initialization**](#Basic-imports-and-initialization)
- [**1. ViT-b-16**](#1.-ViT-b-16)
- [**2. Swin-t**](#2.-Swin-t)
- [**3. MaxViT-t**](#3.-MaxViT-t)

## Basic imports and initialization

$\qquad$ [[Back to top]](#Navigation) $\qquad$ [[Next part $\to$]](#1.-Alexnet)

- [Setting up templates, limiting the hardware resources, importing packages](#Setting-up-templates,-limiting-the-hardware-resources,-importing-packages)
- [Initializing common variables](#Initializing-common-variables)
- [Configuring experiments and generating corresponding sh-files](#Configuring-experiments-and-generating-corresponding-sh-files)

### Setting up templates, limiting the hardware resources, importing packages

$\quad$[[Back to section]](#Basic-imports-and-initialization)$\quad$[[Next subsect.$\to$]](#Initializing-common-variables)

In [1]:
notebook_directory_offset = '../'
sh_file_type = 'base'
libstdcpp_path = ''

In the case of problems with loading ```libstc++.so.6```, please provide the path to the library. In any other case, just ignore the following block.

In [2]:
import ctypes

libstdcpp_path = '/mnt/bulky/pkharyuk/apd/envs/activation_sense/lib/libstdc++.so.6'
try:
    _stdcxx_lib = ctypes.cdll.LoadLibrary(libstdcpp_path)
except:
    pass

To provide easy access to modules stored in the ```../src/``` directory, we use the following workaround:

In [3]:
import os
import sys
sys.path.append(
    os.path.join(notebook_directory_offset, '../src/')
)

Next, we limit the hardware usage by setting the configuration dictionaries, maximum number of threads:

In [4]:
# set limitations on hardware
# fill on the template's config
import hardware_setup

mkl_num_threads = 4
hardware_setup.mkl_set_num_threads(num_threads=mkl_num_threads)

available_gpu_idxs = [0, 2, 3]
max_n_cuda = len(available_gpu_idxs)

[mkl]: set up num_threads=4/4


Then we import all necessary packages:

In [5]:
import copy

import numpy as np

import sensitivity_analysis.augmentation_setting
import sensitivity_analysis.visualize
import sensitivity_analysis.preprocess_visualize

import preparation.single_unit
import preparation.visualize

import exp_assistance

import prediction.compute
import prediction.compute_separated
import prediction.crosscorr
import prediction.hca

import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import display

### Initializing common variables

[[$\leftarrow$Prev.subsect.]](#Setting-up-templates,-limiting-the-hardware-resources,-importing-packages)$\quad$[[Back to section]](#Basic-imports-and-initialization)$\quad$[[Next subsect.$\to$]](#Configuring-experiments-and-generating-corresponding-sh-files)

In [8]:
dataset_name = 'imagenet'
data_dirname = f'../data/{dataset_name}'

sensitivity_values_dirname = '../results/'
model_dirname = '../torch-models/'
scripts_path = '../scripts/'
exp_relative_path = '../experiments/'

values_fnm_base = f'{dataset_name}_values'
output_filename_suffix = 'pred_NOFCL'

results_dirname_path = '../results/'

desired_image_height = 224
desired_image_width = 224

dataset_part = 'valid'

sensitivity_values_name_list = [
    'shpv', 'si', 'siT'
]
no_aug_key = 'original'
y_true_key = 'true_labels'

alphas = (0., 0.5, 1.5)
percentiles = (0.5, 0.6, 0.7, 0.8, 0.9)
inverts = (0, 1)

augmentation_set_numbers_list = (1, 2)

linkage_hca = 'average'

cmap_cormat = 'Reds_r'
cmap_hca = 'twilight_shifted'
label_cmap_hca = 'brg_r'
max_d_label_hca = 0.025
label_background_color_hca = '#0000001A'
label_fontsize_hca = 8

figsizes_cormat = ((5*3+1, 3), (6*3+1, 4))
figsizes_hca = ((3*4+1, 8), (3*4+1, 12))

sensitivity_value_name_list = (
    'shpv',
    'si',
    'siT',
)

In [7]:
basic_config_dict = {
    'recompute_predictions': 1,
    'batch_size_computing': 100,
    'samples_per_class_train': 732,
    'samples_per_class_valid': 50,
    'augmentation_set_number': 3, # 1+2
    'dataset_part': dataset_part,
    'Nsamples': 50*1000,
    'Ninner_samples': 3,
    #'os_environment_config',
    'mkl_num_threads': mkl_num_threads,
    'data_dirname': data_dirname,
    'dataset': dataset_name,
    'model_dirname': model_dirname,
    'desired_image_height': desired_image_height,
    'desired_image_width': desired_image_width,
    'use_permutation_variable': 1,
    'use_class_variable': 1,
    'use_partition_variable': 1,
    'alphas': f'"{alphas}"', #exp_assistance.convert_list2argstr(alphas),
    'percentiles': f'"{percentiles}"', #exp_assistance.convert_list2argstr(percentiles),
    'top_n_predictions': 5,
    'sensitivity_values_dirname': sensitivity_values_dirname,
    'values_fnm_base': values_fnm_base,
    'output_filename_suffix': output_filename_suffix,
}
if len(libstdcpp_path) > 0:
    basic_config_dict['libstdcpp_path'] = libstdcpp_path

### Configuring experiments and generating corresponding sh-files

[[$\leftarrow$Prev.subsect.]](#Initializing-common-variables)$\quad$[[Back to section]](#Basic-imports-and-initialization)$\quad$

- [Alexnet](#Alexnet)
- [VGG11](#VGG11)
- [ResNet18](#ResNet18)


#### ViT-16-b

[[Back to section]](#Configuring-experiments-and-generating-corresponding-sh-files)

In [9]:
vitb16_network_name = 'vit_b_16'
vitb16_model_filename = 'vit_b_16-c867db91.pth'

vitb16_network_modules = [
    'encoder.layers.encoder_layer_2',
    'encoder.layers.encoder_layer_5',
    'encoder.layers.encoder_layer_8',
    'encoder.layers.encoder_layer_11',
    'heads'
]
vitb16_network_modules = exp_assistance.convert_list2argstr(
    vitb16_network_modules
)
vitb16_classification_layer_name = 'heads'

In [10]:
config_dict_vitb16 = copy.deepcopy(basic_config_dict)

config_dict_vitb16['device'] = 'cuda'# manually select visible device in the result
config_dict_vitb16['network_name'] = vitb16_network_name
config_dict_vitb16['network_modules'] = vitb16_network_modules
config_dict_vitb16['classification_layer_name'] = vitb16_classification_layer_name
config_dict_vitb16['subset_random_state_train'] = 297
config_dict_vitb16['subset_random_state_valid'] = 907640
config_dict_vitb16['torch_seed'] = 2340098
config_dict_vitb16['numpy_seed'] = 80042
config_dict_vitb16['class_sampler_seed'] = 102030
config_dict_vitb16['class_selector_seed'] = 981924
config_dict_vitb16['augpar_sampler_seeds'] = 65298
config_dict_vitb16['model_filename'] = vitb16_model_filename

In [12]:
####################################################################################
script_filename_base = f'script_exp3_{vitb16_network_name}_{dataset_name}_{output_filename_suffix}'
exp_filename = '3a_masked_prediction.py'
exp_script_path = os.path.join(exp_relative_path, exp_filename)
for i, sensitivity_value_name in enumerate(sensitivity_values_name_list):
    config_dict_vitb16['results_dirname_path'] = os.path.join(
        results_dirname_path,
        vitb16_network_name,
        sensitivity_value_name
    )
    config_dict_vitb16['sensitivity_values_name'] = sensitivity_value_name
    if config_dict_vitb16['device'].startswith('cuda'):
        config_dict_vitb16['device'] = config_dict_vitb16['device'].split(':')[0]
    current_script_filename = f'{script_filename_base}_sensval={sensitivity_value_name}.sh'
    current_script_path = os.path.join(
        notebook_directory_offset, scripts_path, current_script_filename
    )
    exp_assistance.build_sh_exp_files(
        config_dict_vitb16,
        script_path=current_script_path,
        experiment_file_path=exp_script_path,
        notebook_directory_offset=notebook_directory_offset,
        which=sh_file_type,
    );

#### Swin-t

[[Back to section]](#Configuring-experiments-and-generating-corresponding-sh-files)

In [8]:
swint_network_name = 'swin_t'
swint_model_filename = 'swin_t-704ceda3.pth'

swint_network_modules = [
    'features.1',
    'features.3',
    'features.5',
    'features.7',
    'head'
]
swint_network_modules = exp_assistance.convert_list2argstr(
    swint_network_modules
)
swint_classification_layer_name = 'head'

In [9]:
config_dict_swint = copy.deepcopy(basic_config_dict)

config_dict_swint['device'] = 'cpu' # manually select visible device in the result
config_dict_swint['network_name'] = swint_network_name
config_dict_swint['network_modules'] = swint_network_modules
config_dict_swint['classification_layer_name'] = swint_classification_layer_name
config_dict_swint['subset_random_state_train'] = 6498
config_dict_swint['subset_random_state_valid'] = 4660
config_dict_swint['torch_seed'] = 1598
config_dict_swint['numpy_seed'] = 3689
config_dict_swint['class_sampler_seed'] = 2196
config_dict_swint['class_selector_seed'] = 8580
config_dict_swint['augpar_sampler_seeds'] = 4624
config_dict_swint['model_filename'] = swint_model_filename

config_dict_swint['batch_size_computing'] = 100
config_dict_swint['mkl_num_threads'] = 4

In [10]:
####################################################################################
script_filename_base = f'script_exp3_{swint_network_name}_{dataset_name}_{output_filename_suffix}'
exp_filename = '3a_masked_prediction.py'
exp_script_path = os.path.join(exp_relative_path, exp_filename)
for i, sensitivity_value_name in enumerate(sensitivity_values_name_list):
    config_dict_swint['results_dirname_path'] = os.path.join(
        results_dirname_path,
        swint_network_name,
        sensitivity_value_name
    )
    config_dict_swint['sensitivity_values_name'] = sensitivity_value_name
    if config_dict_swint['device'].startswith('cuda'):
        config_dict_swint['device'] = config_dict_swint['device'].split(':')[0]
        #config_dict_vgg11['device'] += f':{available_gpu_idxs[i%max_n_cuda]}'
    current_script_filename = f'{script_filename_base}_sensval={sensitivity_value_name}.sh'
    current_script_path = os.path.join(
        notebook_directory_offset, scripts_path, current_script_filename
    )
    exp_assistance.build_sh_exp_files(
        config_dict_swint,
        script_path=current_script_path,
        experiment_file_path=exp_script_path,
        notebook_directory_offset=notebook_directory_offset,
        which=sh_file_type,
    );

#### MaxVit-t

[[Back to section]](#Configuring-experiments-and-generating-corresponding-sh-files)

In [11]:
# resnet18
maxvit_t_network_name = 'maxvit_t'
maxvit_t_model_filename = 'maxvit_t-bc5ab103.pth'

maxvit_t_network_modules = [
    'blocks.0',
    'blocks.1',
    'blocks.2',
    'blocks.3',
    #'classifier'
]
maxvit_t_network_modules = exp_assistance.convert_list2argstr(
    maxvit_t_network_modules
)
maxvit_t_classification_layer_name = 'classifier'

In [12]:
config_dict_maxvit_t = copy.deepcopy(basic_config_dict)

config_dict_maxvit_t['device'] = 'cuda' # manually select visible device in the result
config_dict_maxvit_t['network_name'] = maxvit_t_network_name
config_dict_maxvit_t['network_modules'] = maxvit_t_network_modules
config_dict_maxvit_t['classification_layer_name'] = maxvit_t_classification_layer_name
config_dict_maxvit_t['subset_random_state_train'] = 569
config_dict_maxvit_t['subset_random_state_valid'] = 675
config_dict_maxvit_t['torch_seed'] = 3106
config_dict_maxvit_t['numpy_seed'] = 4737
config_dict_maxvit_t['class_sampler_seed'] = 2638
config_dict_maxvit_t['class_selector_seed'] = 2976
config_dict_maxvit_t['augpar_sampler_seeds'] = 3333
config_dict_maxvit_t['model_filename'] = maxvit_t_model_filename

config_dict_maxvit_t['mkl_num_threads'] = 6

In [14]:
####################################################################################
script_filename_base = f'script_exp3_{maxvit_t_network_name}_{dataset_name}_{output_filename_suffix}'
exp_filename = '3a_masked_prediction.py'
exp_script_path = os.path.join(exp_relative_path, exp_filename)
for i, sensitivity_value_name in enumerate(sensitivity_values_name_list):
    config_dict_maxvit_t['results_dirname_path'] = os.path.join(
        results_dirname_path,
        maxvit_t_network_name,
        sensitivity_value_name
    )
    config_dict_maxvit_t['sensitivity_values_name'] = sensitivity_value_name
    if config_dict_maxvit_t['device'].startswith('cuda'):
        config_dict_maxvit_t['device'] = config_dict_maxvit_t['device'].split(':')[0]
        #config_dict_maxvit_t['device'] += f':{available_gpu_idxs[i%max_n_cuda]}'
    current_script_filename = f'{script_filename_base}_sensval={sensitivity_value_name}.sh'
    current_script_path = os.path.join(
        notebook_directory_offset, scripts_path, current_script_filename
    )
    exp_assistance.build_sh_exp_files(
        config_dict_maxvit_t,
        script_path=current_script_path,
        experiment_file_path=exp_script_path,
        notebook_directory_offset=notebook_directory_offset,
        which=sh_file_type,
    );

## 1. ViT-b-16

[[$\leftarrow$ Prev.part]](#Basic-imports-and-initialization) $\qquad$ [[Back to top]](#Navigation) $\qquad$ [[Next part $\to$]](#2.-Swin-t)


- [1.1 Extract accuracy measurements](#1.1-Extract-accuracy-measurements)

### 1.1 Extract accuracy measurements

$\quad$[[Back to section]](#1.-ViT-b-16)

In [15]:
network_name = 'vit_b_16'
value_name = 'si'

featured_measurements_dict = {}
featured_measurements_no_aug_input_dict = {}

results_path = os.path.join(
    notebook_directory_offset,
    results_dirname_path,
    f'{value_name}_{network_name}_{values_fnm_base}_pred_BASIC_part={dataset_part}.hdf5'
)
results_top1, results_topn = prediction.compute.extract_accuracy(
    results_path,
    no_aug_key=no_aug_key,
    y_true_key=y_true_key,
    verbose=True,
)
featured_measurements, featured_measurements_no_aug_input, measurements_no_mask_top_1 = (
    prediction.compute.collect_featured_measurements(
        results_top1,
        augmentation_set_numbers_list,
        alphas=np.empty((0, )),
        percentiles=np.empty((0, )),
        inverts=np.empty((0, )),
        no_aug_key=no_aug_key,
    )
)

_, _, measurements_no_mask_top_n = (
    prediction.compute.collect_featured_measurements(
        results_topn,
        augmentation_set_numbers_list,
        alphas=np.empty((0, )),
        percentiles=np.empty((0, )),
        inverts=np.empty((0, )),
        no_aug_key=no_aug_key,
    )
)
for aug_set_num in augmentation_set_numbers_list:
    print('Top-1 accuracy:')
    display(measurements_no_mask_top_1[aug_set_num])
    print('Top-5 accuracy:')
    display(measurements_no_mask_top_n[aug_set_num])
    
    featured_measurements_dict[aug_set_num] = featured_measurements_dict.get(aug_set_num, {})
    featured_measurements_no_aug_input_dict[aug_set_num] = featured_measurements_no_aug_input_dict.get(
        aug_set_num, {}
    )
    featured_measurements_dict[aug_set_num][value_name] = featured_measurements[aug_set_num]
    featured_measurements_no_aug_input_dict[aug_set_num][value_name] = (
        featured_measurements_no_aug_input[aug_set_num]
    )


si
Nsamples=50000: num.classes=1000, min/mean/max samples per class=50/50.00/50
original::iaug=original top-1 acc=0.81
original::iaug=original top-5 acc=0.95
Top-1 accuracy:


,erasing,sharpness_const,rolling,grayscaling,gaussian_blur,original
0,0.785233,0.8084,0.794667,0.71314,0.763967,0.80978


Top-5 accuracy:


,erasing,sharpness_const,rolling,grayscaling,gaussian_blur,original
0,0.938827,0.95214,0.943287,0.90218,0.929407,0.95266


Top-1 accuracy:


,brightness,contrast,saturation,hue,hflip,rotation,elliptic_local_blur,original
0,0.78584,0.781653,0.792187,0.702867,0.80852,0.777527,0.803007,0.80978


Top-5 accuracy:


,brightness,contrast,saturation,hue,hflip,rotation,elliptic_local_blur,original
0,0.938173,0.9383,0.944493,0.896967,0.95242,0.93496,0.949807,0.95266


## 2. Swin-t

[[$\leftarrow$ Prev.part]](#1.-ViT-b-16) $\qquad$ [[Back to top]](#Navigation) $\qquad$ [[Next part $\to$]](#3.-MaxViT-t)


- [2.1 Extract accuracy measurements](#2.1-Extract-accuracy-measurements)

### 2.1 Extract accuracy measurements

$\quad$[[Back to section]](#2.-Swin-t)

In [16]:
network_name = 'swin_t'
value_name = 'si'

featured_measurements_dict = {}
featured_measurements_no_aug_input_dict = {}

results_path = os.path.join(
    notebook_directory_offset,
    results_dirname_path,
    f'{value_name}_{network_name}_{values_fnm_base}_pred_BASIC_part={dataset_part}.hdf5'
)
results_top1, results_topn = prediction.compute.extract_accuracy(
    results_path,
    no_aug_key=no_aug_key,
    y_true_key=y_true_key,
    verbose=True,
)
featured_measurements, featured_measurements_no_aug_input, measurements_no_mask_top_1 = (
    prediction.compute.collect_featured_measurements(
        results_top1,
        augmentation_set_numbers_list,
        alphas=np.empty((0, )),
        percentiles=np.empty((0, )),
        inverts=np.empty((0, )),
        no_aug_key=no_aug_key,
    )
)

_, _, measurements_no_mask_top_n = (
    prediction.compute.collect_featured_measurements(
        results_topn,
        augmentation_set_numbers_list,
        alphas=np.empty((0, )),
        percentiles=np.empty((0, )),
        inverts=np.empty((0, )),
        no_aug_key=no_aug_key,
    )
)
for aug_set_num in augmentation_set_numbers_list:
    print('Top-1 accuracy:')
    display(measurements_no_mask_top_1[aug_set_num])
    print('Top-5 accuracy:')
    display(measurements_no_mask_top_n[aug_set_num])
    
    featured_measurements_dict[aug_set_num] = featured_measurements_dict.get(aug_set_num, {})
    featured_measurements_no_aug_input_dict[aug_set_num] = featured_measurements_no_aug_input_dict.get(
        aug_set_num, {}
    )
    featured_measurements_dict[aug_set_num][value_name] = featured_measurements[aug_set_num]
    featured_measurements_no_aug_input_dict[aug_set_num][value_name] = (
        featured_measurements_no_aug_input[aug_set_num]
    )


Nsamples=50000: num.classes=1000, min/mean/max samples per class=50/50.00/50
original::iaug=original top-1 acc=0.81
original::iaug=original top-5 acc=0.96
Top-1 accuracy:


,erasing,sharpness_const,rolling,grayscaling,gaussian_blur,original
0,0.790567,0.81226,0.79804,0.75646,0.7318,0.8105


Top-5 accuracy:


,erasing,sharpness_const,rolling,grayscaling,gaussian_blur,original
0,0.945693,0.9558,0.948773,0.92886,0.914213,0.95598


Top-1 accuracy:


,brightness,contrast,saturation,hue,hflip,rotation,elliptic_local_blur,original
0,0.794107,0.790813,0.80206,0.72398,0.81012,0.76326,0.80588,0.8105


Top-5 accuracy:


,brightness,contrast,saturation,hue,hflip,rotation,elliptic_local_blur,original
0,0.946747,0.945173,0.951473,0.911867,0.95596,0.92862,0.95396,0.95598


## 3. MaxViT-t

[[$\leftarrow$ Prev.part]](#2.-Swin-t) $\qquad$ [[Back to top]](#Navigation) $\qquad$


- [3.1 Extract accuracy measurements](#3.1-Extract-accuracy-measurements)

### 3.1 Extract accuracy measurements

$\quad$[[Back to section]](#3.-MaxViT-t)

In [17]:
network_name = 'maxvit_t'
value_name = 'si'

featured_measurements_dict = {}
featured_measurements_no_aug_input_dict = {}

results_path = os.path.join(
    notebook_directory_offset,
    results_dirname_path,
    f'{value_name}_{network_name}_{values_fnm_base}_pred_BASIC_part={dataset_part}.hdf5'
)
results_top1, results_topn = prediction.compute.extract_accuracy(
    results_path,
    no_aug_key=no_aug_key,
    y_true_key=y_true_key,
    verbose=True,
)
featured_measurements, featured_measurements_no_aug_input, measurements_no_mask_top_1 = (
    prediction.compute.collect_featured_measurements(
        results_top1,
        augmentation_set_numbers_list,
        alphas=np.empty((0, )),
        percentiles=np.empty((0, )),
        inverts=np.empty((0, )),
        no_aug_key=no_aug_key,
    )
)

_, _, measurements_no_mask_top_n = (
    prediction.compute.collect_featured_measurements(
        results_topn,
        augmentation_set_numbers_list,
        alphas=np.empty((0, )),
        percentiles=np.empty((0, )),
        inverts=np.empty((0, )),
        no_aug_key=no_aug_key,
    )
)
for aug_set_num in augmentation_set_numbers_list:
    print('Top-1 accuracy:')
    display(measurements_no_mask_top_1[aug_set_num])
    print('Top-5 accuracy:')
    display(measurements_no_mask_top_n[aug_set_num])
    
    featured_measurements_dict[aug_set_num] = featured_measurements_dict.get(aug_set_num, {})
    featured_measurements_no_aug_input_dict[aug_set_num] = featured_measurements_no_aug_input_dict.get(
        aug_set_num, {}
    )
    featured_measurements_dict[aug_set_num][value_name] = featured_measurements[aug_set_num]
    featured_measurements_no_aug_input_dict[aug_set_num][value_name] = (
        featured_measurements_no_aug_input[aug_set_num]
    )


Nsamples=50000: num.classes=1000, min/mean/max samples per class=50/50.00/50
original::iaug=original top-1 acc=0.83
original::iaug=original top-5 acc=0.97
Top-1 accuracy:


,erasing,sharpness_const,rolling,grayscaling,gaussian_blur,original
0,0.81128,0.8353,0.820427,0.79148,0.784567,0.83372


Top-5 accuracy:


,erasing,sharpness_const,rolling,grayscaling,gaussian_blur,original
0,0.954673,0.96666,0.958773,0.9478,0.943767,0.96622


Top-1 accuracy:


,brightness,contrast,saturation,hue,hflip,rotation,elliptic_local_blur,original
0,0.820773,0.819267,0.827073,0.772133,0.83364,0.7857,0.828647,0.83372


Top-5 accuracy:


,brightness,contrast,saturation,hue,hflip,rotation,elliptic_local_blur,original
0,0.959447,0.9587,0.96304,0.939487,0.96626,0.9424,0.96336,0.96622
